# Test FOL Pretrain Model

Load model `Laplaces-Red-Devils/fol-pretrain-malls-qwen2.5-3` va test kha nang sinh FOL tu NL.

**Luu y**: Inference phai dung cung prompt (system + user) nhu luc huan luyen.

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Laplaces-Red-Devils/fol-pretrain-malls-qwen2.5-3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
model.eval()
print(f"Model loaded: {MODEL_ID}")

In [ ]:
# === Prompt giong het luc huan luyen (SYSTEM_PROMPT_FOL_SFT + USER_TEMPLATE_FOL_SFT) ===

SYSTEM_PROMPT = """\
### Instruction
You are a First-Order Logic (FOL) translator. Convert each numbered natural-language (NL) premise into a precise FOL formula by reasoning through the steps below.

### Chain of Thought \u2014 Follow these steps for EACH premise

**Step 1: IDENTIFY THE SUBJECT**
- Named entity (John, the curriculum, Professor Smith) \u2192 treat as CONSTANT, use directly as argument.
- Generic reference (a student, every person, some faculty member) \u2192 treat as VARIABLE, needs a quantifier.
- No subject / standalone fact (The fund is depleted) \u2192 NULLARY predicate (no arguments).

**Step 2: DETERMINE THE QUANTIFIER**
- \"All / Every / Any / If a...\" \u2192 \u2200 (universal).
- \"Some / There exists / At least one...\" \u2192 \u2203 (existential).
- Named entity or standalone fact \u2192 NO quantifier.

**Step 3: CHOOSE PREDICATE NAME**
- If the NL contains a parenthetical hint like (\u00acR), (U), (T) \u2192 use that exact letter.
- Otherwise \u2192 derive a descriptive name from the NL text.
- Entity information belongs in arguments, NOT in the predicate name.
- AVOID over-long predicates that absorb multiple concepts into one name.

**Step 4: BUILD THE LOGICAL STRUCTURE**
- \"If A then B\" \u2192 A \u2192 B
- \"A and B\" \u2192 A \u2227 B
- \"A or B\" \u2192 A \u2228 B
- \"not A\" \u2192 \u00acA
- \"A if and only if B\" \u2192 A \u2194 B

**Step 5: ASSEMBLE THE FORMULA**
- Variable + quantifier: \u2200x (Predicate(x) \u2192 ...)
- Constant (no quantifier): Predicate(John)
- Nullary (no arguments): \u00acdepleted_fund

**Step 6: VALIDATE before outputting**
- n NL premises \u2192 exactly n FOL formulas, same order.
- No invented predicates.
- Constants are NOT quantified.

### Output Format
Output ONLY a JSON object: {\"premises_fol\": [\"...\", \"...\"]}
No markdown fences, no explanation, no commentary outside the JSON.

### Context
- Quantifiers: \u2200x (...), \u2203x (...)
- Connectives: \u2192 (implies), \u2227 (and), \u2228 (or), \u00ac (not), \u2194 (iff)
"""


def generate_fol(premises_nl: list[str], max_new_tokens: int = 512) -> str:
    """Sinh FOL tu NL premises, dung cung prompt nhu luc train."""
    nl_block = "\n".join(f"{i}. {p}" for i, p in enumerate(premises_nl, 1))
    user_msg = f"### Input\n{nl_block}\n\n### Output\n"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
        )

    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return generated


def parse_fol_output(text: str) -> list[str]:
    """Parse JSON output thanh list FOL."""
    import re
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if "premises_fol" in parsed and isinstance(parsed["premises_fol"], list):
                return [str(f).strip() for f in parsed["premises_fol"]]
        except json.JSONDecodeError:
            pass
    return [text]  # fallback


print("Functions ready.")

## Example 1: Learning Principles

In [ ]:
premises_nl_1 = [
    "Procrastination occurs when there is a perceived gap between effort and reward.",
    "If a task has a clear deadline, people are more likely to complete it on time.",
    "If a student uses active recall, they retain more information than passive review.",
    "The Pomodoro technique increases focus by breaking work into timed intervals.",
    "People are more likely to complete a task if they make a public commitment.",
    "Breaking a large task into smaller steps reduces mental resistance.",
    "Sleep is crucial for memory consolidation.",
    "If stress is too high, cognitive performance decreases.",
    "If a student prioritizes urgent tasks over important tasks, long-term learning suffers.",
    "Motivation increases when a person sees progress in their work.",
]

fol_gt_1 = [
    "ForAll(t, perceived_effort_gap(t) \u2192 procrastination(t))",
    "ForAll(t, has_deadline(t) \u2192 more_likely_complete_on_time(t))",
    "ForAll(s, uses_active_recall(s) \u2192 retains_more(s))",
    "ForAll(p, uses_pomodoro(p) \u2192 increased_focus(p))",
    "ForAll(t, public_commitment(t) \u2192 more_likely_complete(t))",
    "ForAll(t, large_task(t) \u2192 reduce_resistance(break_into_steps(t)))",
    "ForAll(s, sufficient_sleep(s) \u2192 better_memory_consolidation(s))",
    "ForAll(s, high_stress(s) \u2192 decreased_cognitive_performance(s))",
    "ForAll(s, prioritizes_urgent(s) \u2192 learning_suffers(s))",
    "ForAll(p, sees_progress(p) \u2192 increased_motivation(p))",
]

print("Generating FOL for Example 1...")
raw_1 = generate_fol(premises_nl_1)
print("\n=== Raw output ===")
print(raw_1)

pred_1 = parse_fol_output(raw_1)
print("\n=== So sanh Predicted vs Ground Truth ===")
for i, (nl, gt) in enumerate(zip(premises_nl_1, fol_gt_1)):
    pred = pred_1[i] if i < len(pred_1) else "(missing)"
    match = "\u2705" if pred.strip() == gt.strip() else "\u274c"
    print(f"\n{i+1}. {nl}")
    print(f"   GT:   {gt}")
    print(f"   PRED: {pred}  {match}")

## Example 2: Aviation Student Regulations

In [ ]:
premises_nl_2 = [
    "If an aviation student completes the aircraft theory course, they are allowed to participate in the flight practice course.",
    "If the weather is bad on the practice day, the flight will be delayed by at least 2 hours.",
    "Students must complete a safety test before flight practice within 1 hour.",
    "If they fail the safety test, students cannot fly and must retake the theory course.",
    "If the flight is delayed by more than 3 hours, students are refunded 50% of the practice course fee.",
    "If students fly more than 1 hour late from the schedule, they must submit a supplementary report.",
    "The flight practice course requires at least 2 instructors: a flight instructor and a technician.",
    "If the flight instructor is absent, the course must find a replacement within 1 hour.",
    "If no replacement is found, the practice session is canceled and students must re-register.",
    "Students must submit their practice registration application 7 days in advance for scheduling.",
    "Lan completed the aircraft theory course on March 20, 2025.",
    "Bad weather was reported at 8:00 AM on the practice day, March 25, 2025.",
    "Lan passed the safety test at 7:30 AM on the same day.",
    "The flight instructor was absent at 8:15 AM on the practice day.",
    "No replacement flight instructor was found within 1 hour.",
    "Students must be notified of the practice session cancellation at least 30 minutes before the scheduled 9:00 AM flight.",
    "If the practice session is canceled, students must take a make-up session within 2 weeks or lose their flight exam eligibility.",
]

fol_gt_2 = [
    "\u2200s (complete_theory(s) \u2192 participate_practice(s))",
    "\u2200s (bad_weather(s) \u2192 delay_practice(s) \u2265 2)",
    "\u2200s (flight_practice(s) \u2192 safety_test(s) \u2264 1)",
    "\u2200s (\u00acsafety_test(s) \u2192 \u00acflight_practice(s) \u2227 retake_theory(s))",
    "\u2200s (delay_practice(s) > 3 \u2192 refund(s) = 0.5)",
    "\u2200s (flight_late(s) > 1 \u2192 submit_report(s))",
    "\u2200s (flight_practice(s) \u2192 instructors(s) \u2265 2)",
    "\u2200s (instructor_absent(s) \u2192 find_replacement(s) \u2264 1)",
    "\u2200s (instructor_absent(s) \u2227 \u00acfind_replacement(s) \u2192 cancel_practice(s) \u2227 reregister(s))",
    "\u2200s (flight_practice(s) \u2192 submit_application(s) \u2265 7)",
    "complete_theory(Lan, '20/3/2025')",
    "bad_weather(Lan, '8:00', '25/3/2025')",
    "safety_test(Lan, '7:30', '25/3/2025')",
    "instructor_absent(Lan, '8:15', '25/3/2025')",
    "\u00acfind_replacement(Lan, 1)",
    "\u2200s (cancel_practice(s) \u2192 notify_cancellation(s) \u2265 0.5)",
    "\u2200s (cancel_practice(s) \u2192 make_up_session(s) \u2264 2 \u2228 lose_exam_eligibility(s))",
]

print("Generating FOL for Example 2...")
raw_2 = generate_fol(premises_nl_2, max_new_tokens=768)
print("\n=== Raw output ===")
print(raw_2)

pred_2 = parse_fol_output(raw_2)
print("\n=== So sanh Predicted vs Ground Truth ===")
for i, (nl, gt) in enumerate(zip(premises_nl_2, fol_gt_2)):
    pred = pred_2[i] if i < len(pred_2) else "(missing)"
    match = "\u2705" if pred.strip() == gt.strip() else "\u274c"
    print(f"\n{i+1}. {nl}")
    print(f"   GT:   {gt}")
    print(f"   PRED: {pred}  {match}")

## Summary

In [ ]:
total = len(fol_gt_1) + len(fol_gt_2)
exact = 0
for gt, pred_list in [(fol_gt_1, pred_1), (fol_gt_2, pred_2)]:
    for i, g in enumerate(gt):
        p = pred_list[i] if i < len(pred_list) else ""
        if p.strip() == g.strip():
            exact += 1

print(f"Exact match: {exact}/{total} = {exact/total:.2%}")
print(f"Generated count Ex1: {len(pred_1)}/{len(fol_gt_1)}")
print(f"Generated count Ex2: {len(pred_2)}/{len(fol_gt_2)}")
print()
print("Luu y: Exact match thap la binh thuong vi:")
print("- Predicate naming co the khac (CamelCase vs snake_case)")
print("- Cau truc logic tuong duong nhung viet khac")
print("- Quan trong la LOGIC dung, khong nhat thiet phai giong chu")